# Pneumonia Detection from Chest X-Rays

End-to-end PyTorch portfolio notebook using **ResNet18 transfer learning**.

> Educational project only. Not clinically validated and not intended for diagnosis.

The repository contains `demo_results/` only to show what the final presentation should look like. The cells below generate the **real outputs** once you point `DATA_DIR` to the actual Kaggle dataset.


In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch CUDA runtime:", torch.version.cuda)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


## 1. Point the notebook to the Kaggle dataset


In [ ]:
# Change this to your local chest_xray folder.
DATA_DIR = Path(r"C:\path\to\chest_xray")

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    print(p, "exists:", p.exists())


## 2. Image transforms


In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


## 3. Load datasets and DataLoaders


In [ ]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=eval_tfms)
test_ds  = datasets.ImageFolder(TEST_DIR, transform=eval_tfms)

print("Classes:", train_ds.classes)
print("Class mapping:", train_ds.class_to_idx)
print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


## 4. Inspect class distribution


In [ ]:
def class_counts(ds):
    counts = np.bincount(ds.targets)
    return {ds.classes[i]: int(counts[i]) for i in range(len(counts))}

print("Train distribution:", class_counts(train_ds))
print("Validation distribution:", class_counts(val_ds))
print("Test distribution:", class_counts(test_ds))


## 5. Visualize several training images


In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

indices = [0, max(1, len(train_ds)//6), max(2, len(train_ds)//3), max(3, len(train_ds)//2)]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, idx in zip(axes, indices):
    img, label = train_ds[idx]
    img = (img * std + mean).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(train_ds.classes[label])
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_xrays.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Build the ResNet18 transfer-learning model


In [ ]:
weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Baseline: freeze pretrained feature extractor.
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

print(model.fc)


## 7. Loss, optimizer, and training helper


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

def run_epoch(model, loader, train_mode=False):
    model.train() if train_mode else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, labels)

            if train_mode:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


## 8. Train and validate


In [ ]:
EPOCHS = 5
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_acc = -1.0
best_path = Path("best_resnet18_pneumonia.pth")

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(model, val_loader, train_mode=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print("  Saved new best model")

print("Best validation accuracy:", best_val_acc)


## 9. Save training curves


In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(7, 4.5))
plt.plot(epochs, history["train_loss"], marker="o", label="Training loss")
plt.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.grid(True, alpha=0.25)
plt.legend()
plt.savefig(RESULTS_DIR / "training_loss.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 4.5))
plt.plot(epochs, history["train_acc"], marker="o", label="Training accuracy")
plt.plot(epochs, history["val_acc"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.grid(True, alpha=0.25)
plt.legend()
plt.savefig(RESULTS_DIR / "training_accuracy.png", dpi=180, bbox_inches="tight")
plt.show()


## 10. Load the best checkpoint and evaluate the held-out test set


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)

metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
}

for k, v in metrics.items():
    print(f"{k:10s}: {v:.4f}")

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)


## 11. Confusion matrix


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.xticks([0, 1], ["Normal", "Pneumonia"])
plt.yticks([0, 1], ["Normal", "Pneumonia"])
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


## 12. Save a grid of sample predictions


In [ ]:
class_names = test_ds.classes
sample_indices = np.linspace(0, len(test_ds)-1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for ax, idx in zip(axes, sample_indices):
    image_tensor, true_label = test_ds[idx]

    with torch.no_grad():
        logits = model(image_tensor.unsqueeze(0).to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        pred_label = probs.argmax(dim=1).item()
        confidence = probs[0, pred_label].item()

    img = (image_tensor * std + mean).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(
        f"Actual: {class_names[true_label]}\n"
        f"Pred: {class_names[pred_label]} ({confidence:.1%})"
    )

plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_predictions.png", dpi=180, bbox_inches="tight")
plt.show()


## 13. Error-analysis helper

Use this section to inspect **false negatives** and **false positives** rather than reporting metrics only.


In [ ]:
errors = []
for idx, (truth, pred, prob) in enumerate(zip(y_true, y_pred, y_prob)):
    if truth != pred:
        errors.append((idx, truth, pred, prob))

print("Total test errors:", len(errors))
print("First 10:", errors[:10])


## 14. Optional Grad-CAM — conceptual implementation

Grad-CAM is useful for model inspection, but it is **not a clinical explanation**. The code below provides a minimal hook-based implementation for the final convolutional block.


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._save_activation)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, image_tensor, class_idx=None):
        self.model.zero_grad()
        logits = self.model(image_tensor)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        score = logits[:, class_idx].sum()
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy(), class_idx

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

# Example after training:
# gradcam = GradCAM(model, model.layer4[-1].conv2)
# x, label = test_ds[0]
# heatmap, pred_class = gradcam.generate(x.unsqueeze(0).to(DEVICE))
# gradcam.close()


## 15. Final portfolio checklist

After the real run:

1. Replace all `demo_results/` references in the README with real `results/` images.
2. Insert your measured Accuracy, Precision, Recall, F1 and ROC-AUC.
3. Mention at least one limitation and summarize the main error types.
4. Keep the medical disclaimer.
5. Push the completed repository to GitHub.
6. Use the copy in `LINKEDIN_COPY.md` for LinkedIn Featured and your project post.
